# Import

In [2]:
from scipy.io import loadmat
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
from mapd import Trial, Table, Sinq
import h5py

from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
%load_ext autoreload
%autoreload 2
%matplotlib widget

def refresh_table_addons():
    import importlib
    import mapd.table_plotters as tps
    import mapd.table_scalars as tbs
    importlib.reload(tps)
    importlib.reload(tbs)

    for name in dir(tps):
        if name.startswith('_'):
            continue
        attr = getattr(tps, name)
        setattr(Table, name[len("plot_"):], attr)
    
    for name in dir(tbs):
        
        if name.startswith('_'):
            continue
        if name.startswith('compute_'):
            attr = getattr(tbs, name)
            setattr(Table, name[len("compute_"):], attr)


RECOMPUTE = False

# Load dataset 2 controls sinq

In [3]:
sinq = Sinq(sinqname='dataset_2_controls')
print(sinq.__repr__())

Found data directory: D:\Data
Sinq(dataset_2_controls, 39 rows x 33 params): 0 empties; ['241029_F2_C1', '241029_F3_C1', '241101_F1_C1', '241101_F2_C1', '241105_F1_C1', '241105_F2_C1', '241105_F3_C1', '250813_F1_C1', '250813_F3_C1', '251103_F1_C1', '251103_F2_C1', '251105_F1_C1', '251107_F1_C1', '251107_F2_C1', '251113_F1_C1', '251113_F2_C1', '251124_F1_C1', '251126_F1_C1', '251203_F1_C1', '251203_F2_C1', '251205_F1_C1', '251205_F2_C1', '251210_F1_C1', '251210_F2_C1', '251216_F1_C1', '251216_F2_C1', '251216_F3_C1', '251217_F1_C1', '251218_F1_C1', '251218_F2_C1', '251218_F3_C1', '251219_F1_C2', '260603_F2_C1', '260603_F3_C1', '260619_F1_C1', '260619_F2_C1', '260623_F1_C1', '260623_F2_C1', '260623_F3_C1'] x ['Table', 'parquet', 'genotype', 'notes', 'duration', 'rms_velocity', 'outcome_fractions_no_as_no_mv', 'outcome_fractions_as_off', 'outcome_fractions_rest', 'outcome_fractions_no_as_mv', 'outcome_fractions_probe', 'outcome_fractions_as_off_late', 'outcome_fractions_timeout_fail', 'out

In [5]:
sinq.display_df()

,Table,parquet,genotype,notes,note_text,tags,duration,rms_velocity,outcome_fractions_no_as_no_mv,outcome_fractions_as_off,...,most_common_fiberLED,lo_target_off_state,hi_target_off_state,probe_positive_effort,successes,hard_successes,holding_cost,rest_time_fraction,drift_time_fraction,move_time_fraction
241029_F2_C1,None,LEDFlashTriggerPiezoControl_241029_F2_C1_Table...,ppk-Gal4;10XUAS-ChR in WT,"{'text': 'VNC cut', 'tags': ['control', 'vnc_c...",VNC cut,"control, vnc_cut",3353.03808,6.542378e+04,0.091772,0.313291,...,625_red,0.364162,0.044190,2.252077e+06,26.0,7.0,2.829190e+06,0.696190,0.070104,0.233707
241029_F3_C1,None,LEDFlashTriggerPiezoControl_241029_F3_C1_Table...,+;ppk-GAL4;10XUAS-ChR,"{'text': 'VNC cut', 'tags': ['control', 'vnc_c...",VNC cut,"control, vnc_cut",3946.73730,1.807070e+05,0.067164,0.286070,...,625_red,0.067024,0.004101,4.219859e+06,10.0,4.0,1.561315e+06,0.776807,0.069308,0.153885
241101_F1_C1,None,LEDFlashTriggerPiezoControl_241101_F1_C1_Table...,+;ppk-GAL4;10XUAS-ChR,"{'text': 'VNC cut', 'tags': ['control', 'vnc_c...",VNC cut,"control, vnc_cut",3382.94854,6.515568e+04,0.050000,0.200000,...,625_red,0.122721,0.065017,2.023283e+06,25.0,2.0,3.557914e+06,0.691011,0.119709,0.189280
241101_F2_C1,None,LEDFlashTriggerPiezoControl_241101_F2_C1_Table...,+;ppk-GAL4;10XUAS-ChR,"{'text': 'VNC cut', 'tags': ['control', 'vnc_c...",VNC cut,"control, vnc_cut",3841.91018,4.881359e+04,0.183623,0.196030,...,625_red,0.170548,0.058861,1.047571e+06,21.0,9.0,2.751431e+06,0.797394,0.077439,0.125167
241105_F1_C1,None,LEDFlashTriggerPiezoControl_241105_F1_C1_Table...,+;ppk-GAL4;10XUAS-ChR,"{'text': 'VNC intact', 'tags': ['control', 'vn...",VNC intact,"control, vnc_cut",5556.31728,1.170114e+05,0.347909,0.186312,...,625_red,0.105621,0.073137,3.487105e+06,22.0,10.0,5.820581e+06,0.717468,0.089123,0.193409
241105_F2_C1,None,LEDFlashTriggerPiezoControl_241105_F2_C1_Table...,+;Hot-cell-GAL4;10XUAS-ChR,"{'text': 'VNC cut', 'tags': ['control', 'vnc_c...",VNC cut,"control, vnc_cut",4819.89502,1.049747e+05,0.112867,0.241535,...,625_red,0.217563,0.006419,2.196516e+06,23.0,10.0,2.225910e+06,0.778770,0.032245,0.188985
241105_F3_C1,None,LEDFlashTriggerPiezoControl_241105_F3_C1_Table...,+;Hot-cell-GAL4;10XUAS-ChR,"{'text': 'VNC cut', 'tags': ['control', 'vnc_c...",VNC cut,"control, vnc_cut",3112.96220,9.829863e+04,0.114035,0.432749,...,625_red,0.074885,0.070511,2.722215e+06,23.0,6.0,5.562356e+06,0.648425,0.061192,0.290382
250813_F1_C1,None,LEDFlashTriggerPiezoControl_250813_F1_C1_Table...,"w,NorpA","{'text': '', 'tags': ['control', 'norpa'], 'au...",,"control, norpa",8825.12850,1.533336e+05,0.269341,0.133238,...,epi_only,0.516409,0.166720,4.897259e+06,52.0,18.0,1.316828e+07,0.652911,0.218908,0.128181
250813_F3_C1,None,LEDFlashTriggerPiezoControl_250813_F3_C1_Table...,"w,NorpA","{'text': '', 'tags': ['control', 'norpa'], 'au...",,"control, norpa",9570.74166,2.273700e+05,0.143038,0.221519,...,epi_only,0.349751,0.133878,1.125018e+07,82.0,20.0,2.228913e+07,0.682092,0.078694,0.239214
251103_F1_C1,None,LEDFlashTriggerPiezoControl_251103_F1_C1_Table...,+;pJFRC7;+,"{'text': 'Constant 1s stim', 'tags': ['1s_stim...",Constant 1s stim,"1s_stim, control",6630.34130,3.494666e+05,0.000000,0.000000,...,epi_only,0.074640,0.037451,1.173648e+07,0.0,0.0,8.110773e+06,0.678876,0.037960,0.283164


In [ ]:
# sinq.restore_table(sinq.df.index[0])
# sinq.add_column('rest_time_fraction')
# sinq.add_column('drift_time_fraction')
# sinq.add_column('move_time_fraction')
# sinq.df.loc[sinq.df.index[0]]

Found data directory: D:\Data
T = pd.read_parquet("D:\\Data\\241029\\241029_F2_C1\\LEDFlashTriggerPiezoControl_241029_F2_C1_Table.parquet")
Getting trials
Excluding trials:
[Trial(trial=149, 241029_F2_C1, dT=828.80958, ex=True)]
Getting all meta keys
['as_outcome', 'fiberLED', 'filtercube_status', 'vnc_status']
Found 4 target positions - Counter({('lo', 730.0, 550.0, -180.0, 60.0): 126, ('hi', 730.0, 450.0, -280.0, 60.0): 75, ('lo', 740.0, 560.0, -180.0, 60.0): 63, ('hi', 740.0, 460.0, -280.0, 60.0): 52})


move_time_fraction    0.233707
Name: 241029_F2_C1, dtype: object

In [ ]:
# cols2add = sinq1.df.columns[~sinq1.df.columns.isin(sinq.df.columns)]
# # cols2add = sinq1.df.columns.difference(sinq.df.columns)
# cols2add
# # 2. Add them to sinq.df with pd.NA
# for col in cols2add:
#     sinq.df[col] = np.nan

In [ ]:
# sinq.sync()

In [ ]:
# Standard compute
def run_compute(sinq,id):
    if id in sinq.df.index:
        T = sinq.restore_table(dayflycell=id,overwrite=True)
        # endofdaytrials = T.df.loc[1081:]
        # T.exclude_list_of_trials(endofdaytrials.index,reason='tired, frustrated poor behavior')
    else:
        T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
        T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
        T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
        sinq.add_table(table=T)
        sinq.df
    T.exclude_trials()
    T.extract_trial_properties()
    T.find_successful_trials()
    return sinq,T


# Standard plotting
def run_plots(T):
    fig,ax = T.plot_outcomes(savefig=True,format='png')
    fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')

    df_filters = {'all':None,
              'hi':{'pyasState':'hi'},
              'lo':{'pyasState':'lo'},}

    for filt_key in df_filters.keys():
        dff = df_filters[filt_key]
        fig,ax = T.plot_probe_distribution(binwidth=2,bin_min=-500,bin_max=10,
                                df_filter=dff,index=None,savefig=False,format='png',
                                export=True)

# Constant 1 s stim. 

## 251103_F1_C1

In [ ]:
id = '251103_F1_C1'
# sinq.clear_note(dayflycell=id)
sinq.set_note(
        dayflycell = id,
        text= 'Constant 1s stim',
        tags=['control','1s_stim'],
        append=False,
)
if RECOMPUTE:
    sinq,T = run_compute(sinq,id)
    run_plots(T)

## 251103_F2_C1

In [ ]:
id = '251103_F2_C1'
# sinq.clear_note(dayflycell=id)
sinq.set_note(
        dayflycell = id,
        text= 'Constant 1s stim',
        tags=['control','1s_stim'],
        append=False,
)
if RECOMPUTE:
    sinq,T = run_compute(sinq,id)
    run_plots(T)

In [ ]:
# T.extract_trial_properties()
# T.df.loc[T.df.as_duration == 0,'is_rest']
# fig,ax = T.plot_some_probe_groups(index=T.df.loc[45:49].index)
# display(fig)

# fig = Figure(figsize=(6, 6), dpi=200)
# ax = fig.add_subplot(111)

# T.plot_some_probe_groups(index=T.df.loc[45:49].index,ax=ax) #,force_pos=True)
# display(fig)
# fig.savefig(f'./Figure2/250304_F3_C1/long_sequences/trial_1262.svg')

## 251107_F1_C1

In [ ]:
id = '251107_F1_C1'
sinq.set_note(
        dayflycell = id,
        text= 'Constant 1s stim',
        tags=['control','1s_stim'],
        append=False,
)
if RECOMPUTE:
    sinq,T = run_compute(sinq,id)
    run_plots(T)

## 251107_F2_C1

In [ ]:
id = '251107_F2_C1'
sinq.set_note(
    dayflycell = id,
    text= 'Constant 1s stim',
    tags=['control','learn','1s_stim'],
    append=False,
)
if RECOMPUTE:
    sinq,T = run_compute(sinq,id)
    run_plots(T)

# endofdaytrials = T.df.loc[662:]
# T.exclude_list_of_trials(endofdaytrials.index,reason='out of focus')
# T.exclude_trials()
# T.extract_trial_properties()
# T.find_successful_trials()

# sinq.add_table(table = T)


In [ ]:
sinq.display_df()

In [ ]:
fig = Figure(figsize=(6, 6), dpi=200)
ax = fig.add_subplot(111)

T.plot_some_probe_groups(index=T.df.loc[660:670].index,ax=ax) #,force_pos=True)
display(fig)
# fig.savefig(f'./Figure2/250304_F3_C1/long_sequences/trial_1262.svg')

# Replay

## 251113_F1_C1

In [ ]:
id = '251113_F1_C1'
sinq.set_note(
    dayflycell = id,
    text= 'Replay',
    tags=['control','replay','cumulative_punishment_250228_F2_C1.mat'],
)
if RECOMPUTE:
    sinq,T = run_compute(sinq,id)
    run_plots(T)

## 251113_F2_C1

In [ ]:
id = '251113_F2_C1'
sinq.set_note(
    dayflycell = id,
    text= 'Replay',
    tags=['control','replay','cumulative_punishment_250228_F2_C1.mat'],
)
if RECOMPUTE:
    sinq,T = run_compute(sinq,id)
    run_plots(T)
    sinq.drop_tables()

## 251126_F1_C1

In [ ]:
id = '251126_F1_C1'
sinq.set_note(
    dayflycell = id,
    text= 'Replay',
    tags=['control','replay','cumulative_punishment_250228_F2_C1.mat'],
)
if RECOMPUTE:
    sinq,T = run_compute(sinq,id)
    run_plots(T)
    sinq.drop_tables()

In [ ]:
sinq.display_df(show_tags=True).loc['251113_F1_C1','tags']


## 251203_F1_C1

In [ ]:
id = '251203_F1_C1'

if RECOMPUTE:
    sinq,T = run_compute(sinq,id)
    run_plots(T)
    sinq.drop_tables()
sinq.set_note(
    dayflycell = id,
    text= 'Replay',
    tags=['control','replay','cumulative_punishment_250228_F2_C1.mat'],
)

## 251203_F2_C1

In [ ]:
id = '251203_F2_C1'
if RECOMPUTE:
    sinq,T = run_compute(sinq,id)
    run_plots(T)
    sinq.drop_tables()

sinq.set_note(
    dayflycell = id,
    text= 'Replay',
    tags=['control','replay','cumulative_punishment_250228_F2_C1.mat'],
)

## 251205_F1_C1

In [ ]:
id = '251205_F1_C1'
if RECOMPUTE:
    sinq,T = run_compute(sinq,id)
    run_plots(T)
    sinq.drop_tables()
sinq.set_note(
    dayflycell = id,
    text= 'Replay',
    tags=['control','replay','cumulative_punishment_250228_F2_C1.mat'],
)

## 251205_F2_C1

In [ ]:
id = '251205_F2_C1'
if RECOMPUTE:
    sinq,T = run_compute(sinq,id)
    run_plots(T)
    sinq.drop_tables()
sinq.set_note(
    dayflycell = id,
    text= 'Replay',
    tags=['control','replay','cumulative_punishment_250228_F2_C1.mat'],
)

# No probe feedback

## 251124_F1_C1

In [ ]:
id = '251124_F1_C1'

if RECOMPUTE:
    sinq,T = run_compute(sinq,id)
    run_plots(T)
    sinq.drop_tables()

sinq.clear_note(dayflycell=id)
sinq.set_note(
    dayflycell = id,
    text= 'No probe feedback',
    tags=['control','no_cue'],
)

# WT no antenna

## 251217_F1_C1



In [ ]:
id = '251217_F1_C1'

if RECOMPUTE:
    sinq,T = run_compute(sinq,id)
    run_plots(T)
    sinq.drop_tables()
sinq.clear_note
sinq.set_note(
    dayflycell = id,
    text= 'No antenna',
    tags=['control','no_antenna'],
)

## 251218_F1_C1



In [ ]:
id = '251218_F1_C1'

if RECOMPUTE:
    sinq,T = run_compute(sinq,id)
    run_plots(T)
    sinq.drop_tables()

sinq.set_note(
    dayflycell = id,
    text= 'No antenna',
    tags=['control','no_antenna'],
)

In [ ]:
sinq.display_df()

## 251218_F3_C1

Wild type fly, but again the femur is freee. Fuck...
But I wonder. Does that change the the ability of the lfy to perform the task? I think it could, if there are more degrees of freedom
that the fly has to figure out, it could be part of the problem for the fly.
I'm stopping this one after 350, but it still proves the point, likely.? Doesn't learn, so it's a little confusing.
Certainly different than IAV, which doesn't even try to hold.


In [ ]:
id = '251218_F3_C1'
if RECOMPUTE:
    sinq,T = run_compute(sinq,id)
    run_plots(T)
    sinq.drop_tables()
sinq.set_note(
    dayflycell = id,
    text= 'No antenna',
    tags=['control','no_antenna'],
)

# NorpA

## 250813_F1_C1

In [ ]:
sinq.df

In [ ]:
id = '250813_F1_C1'
if RECOMPUTE:
    sinq,T = run_compute(sinq,id)
    run_plots(T)
    sinq.drop_tables()
sinq.set_note(
    dayflycell = id,
    text= '',
    tags=['control','norpa'],
)

## 250813_F3_C1

In [ ]:
id = '250813_F3_C1'
if RECOMPUTE:
    sinq,T = run_compute(sinq,id)
    run_plots(T)
    sinq.drop_tables()

sinq.set_note(
    dayflycell = id,
    text= '',
    tags=['control','norpa'],
)

## 251210_F1_C1

In [ ]:
id = '251210_F1_C1'
if RECOMPUTE:
    sinq,T = run_compute(sinq,id)
    run_plots(T)
    sinq.drop_tables()

sinq.set_note(
    dayflycell = id,
    text= '',
    tags=['control','norpa'],
)

## 251210_F2_C1

In [ ]:
id = '251210_F2_C1'

# sinq,T = run_compute(sinq,id)
# # solution_ran_out = T.df.loc[421:]
# # solution_ran_out.index
# # T.exclude_list_of_trials(solution_ran_out.index,'solution_ran_out')
# # T.exclude_trials()

# # solution_ran_out = T.df.loc[142:155]
# # solution_ran_out.index
# # T.exclude_list_of_trials(solution_ran_out.index,'kicking, not flexing')
# # T.exclude_trials()

# run_plots(T)
if RECOMPUTE:
    sinq,T = run_compute(sinq,id)
    run_plots(T)
    sinq.drop_tables()

sinq.set_note(
    dayflycell = id,
    text= '',
    tags=['control','norpa'],
)

# NorpA/antennaless

## 251216_F1_C1

In [ ]:
id = '251216_F1_C1'
if RECOMPUTE:
    sinq,T = run_compute(sinq,id)
    run_plots(T)
    sinq.drop_tables()

sinq.set_note(
    dayflycell = id,
    text= 'No antenna',
    tags=['control','norpa','no_antenna'],
)

## 251216_F2_C1

In [ ]:
id = '251216_F2_C1'
if RECOMPUTE:
    sinq,T = run_compute(sinq,id)
    run_plots(T)
    sinq.drop_tables()

sinq.set_note(
    dayflycell = id,
    text= 'No antenna',
    tags=['control','norpa','no_antenna'],
)

## 251216_F3_C1

In [ ]:
id = '251216_F3_C1'
if RECOMPUTE:
    sinq,T = run_compute(sinq,id)
    run_plots(T)
    sinq.drop_tables()

sinq.set_note(
    dayflycell = id,
    text= 'No antenna',
    tags=['control','norpa','no_antenna'],
)

## 251218_F2_C1

In [ ]:
id = '251218_F2_C1'
if RECOMPUTE:
    sinq,T = run_compute(sinq,id)
    run_plots(T)
    sinq.drop_tables()

sinq.set_note(
    dayflycell = id,
    text= 'No antenna',
    tags=['control','norpa','no_antenna'],
)

## 260603_F2_C1

In [ ]:
RECOMPUTE = False

In [ ]:
id = '260603_F2_C1'
if RECOMPUTE:
    sinq,T = run_compute(sinq,id)
# T.assign_column_value('antenna_state','intact',trial_min=1,trial_max=100,write_to_hdf5=True)
# T.assign_column_value('antenna_state','arista_cut_wleft_a3',trial_min=101,trial_max=361,write_to_hdf5=True)
# T.assign_column_value('antenna_state','a3_cut',trial_min=362,trial_max=460,write_to_hdf5=True)
# T.assign_column_value('antenna_state','a1_a2_cut',trial_min=461,write_to_hdf5=True)
run_plots(T)
sinq.drop_tables()

sinq.set_note(
    dayflycell = id,
    text= 'No antenna',
    tags=['control','norpa','no_antenna'],
)

## 260603_F3_C1

In [ ]:
RECOMPUTE = True

In [ ]:
id = '260603_F3_C1'
if RECOMPUTE:
    sinq,T = run_compute(sinq,id)
    # T.assign_column_value('antenna_state','intact',trial_min=1,trial_max=101,write_to_hdf5=True)
    # T.assign_column_value('antenna_state','arista_cut',trial_min=102,trial_max=203,write_to_hdf5=True)
    # T.assign_column_value('antenna_state','a3_cut',trial_min=204,trial_max=303,write_to_hdf5=True)
    # T.assign_column_value('antenna_state','a1_a2_cut',trial_min=304,write_to_hdf5=True)
    run_plots(T)
    sinq.drop_tables()

sinq.set_note(
    dayflycell = id,
    text= 'No antenna',
    tags=['control','norpa','no_antenna'],
)

In [ ]:
T.genotype

In [ ]:
sinq.display_df()

# norpA/+;ss,cry/+

## 251105_F1_C1

In [ ]:
id = '251105_F1_C1'
if RECOMPUTE:
    sinq,T = run_compute(sinq,id)
    run_plots(T)
    sinq.drop_tables()

sinq.set_note(
    dayflycell = id,
    text= '+,norpA/+;ss,cry/+',
    tags=['control','learn'],
)

# WT easier targets

## 251219_F1_C2

In [ ]:
id = '251219_F1_C2'
if RECOMPUTE:
    sinq,T = run_compute(sinq,id)
    run_plots(T)
    sinq.drop_tables()

sinq.set_note(
    dayflycell = id,
    text= 'Easier targets',
    tags=['learn','easy_targets'],
)

# PPK-Gal4

## 241029_F2_C1 -  ppk-Gal4>Chr in WT

In [ ]:
id = '241029_F2_C1'

# Special compute
def run_compute(sinq,id):
    if id in sinq.df.index:
        T = sinq.restore_table(dayflycell=id,overwrite=False)
    else:
        T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
        T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
        T.assign_column_value('fiberLED','625_red',trial_min =1,write_to_hdf5=True)
        T.assign_column_value('vnc_status','intact',trial_min =1,trial_max =149,write_to_hdf5=True)
        T.assign_column_value('vnc_status','cut',trial_min =150,write_to_hdf5=True)
        sinq.add_table(table=T)
    T.exclude_trials()
    T.extract_trial_properties()
    T.find_successful_trials()
    return sinq,T

def run_plots(T):
    fig,ax = T.plot_outcomes(savefig=True,format='png')
    fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')

    df_filters = {'all':None,
              'hi_intact':{'pyasState':'hi','vnc_status':'intact'},
              'lo_intact':{'pyasState':'lo','vnc_status':'intact'},
              'hi_cut':{'pyasState':'hi','vnc_status':'cut'},
              'lo_cut':{'pyasState':'lo','vnc_status':'cut'},
    }

    for filt_key in df_filters.keys():
        dff = df_filters[filt_key]
        fig,ax = T.plot_probe_distribution(binwidth=2,bin_min=-500,bin_max=10,
                                df_filter=dff,index=None,savefig=False,format='png',
                                export=True)
        

if RECOMPUTE:
    sinq,T = run_compute(sinq,id)
    run_plots(T)
    sinq.drop_tables()

sinq.set_note(
    dayflycell = id,
    text= 'VNC cut',
    tags=['control','vnc_cut'],
)

## 241029_F3_C1 -  ppk-Gal4>Chr in WT

In [ ]:
id = '241029_F3_C1'

# Special compute
def run_compute(sinq,id):
    if id in sinq.df.index:
        T = sinq.restore_table(dayflycell=id,overwrite=True)
    else:
        T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
        T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
        T.assign_column_value('fiberLED','625_red',trial_min =1,write_to_hdf5=True)
        T.assign_column_value('vnc_status','intact',trial_min =1,trial_max =207,write_to_hdf5=True)
        T.assign_column_value('vnc_status','cut',trial_min =208,write_to_hdf5=True)
        sinq.add_table(table=T)
        
    T.exclude_trials()
    T.extract_trial_properties()
    T.find_successful_trials()
    return sinq,T

def run_plots(T):
    fig,ax = T.plot_outcomes(savefig=True,format='png')
    fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')

    df_filters = {'all':None,
              'hi_intact':{'pyasState':'hi','vnc_status':'intact'},
              'lo_intact':{'pyasState':'lo','vnc_status':'intact'},
              'hi_cut':{'pyasState':'hi','vnc_status':'cut'},
              'lo_cut':{'pyasState':'lo','vnc_status':'cut'},
    }


    for filt_key in df_filters.keys():
        dff = df_filters[filt_key]
        fig,ax = T.plot_probe_distribution(binwidth=2,bin_min=-500,bin_max=10,
                                df_filter=dff,index=T.df.loc[208:].index,savefig=False,format='png',
                                export=True)


if RECOMPUTE:
    sinq,T = run_compute(sinq,id)
    run_plots(T)
    sinq.drop_tables()

sinq.set_note(
    dayflycell = id,
    text= 'VNC cut',
    tags=['control','vnc_cut'],
)

## 241101_F1_C1 -  ppk-Gal4>Chr in WT

In [ ]:
id = '241101_F1_C1'

# Special compute
def run_compute(sinq,id):
    if id in sinq.df.index:
        T = sinq.restore_table(dayflycell=id,overwrite=True)
    else:
        T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
        T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
        T.assign_column_value('fiberLED','625_red',trial_min =1,write_to_hdf5=True)
        T.assign_column_value('vnc_status','intact',trial_min =1,trial_max =208,write_to_hdf5=True)
        T.assign_column_value('vnc_status','cut',trial_min =209,write_to_hdf5=True)
        sinq.add_table(table=T)
    T.exclude_trials()
    T.extract_trial_properties()
    T.find_successful_trials()
    return sinq,T

def run_plots(T):
    fig,ax = T.plot_outcomes(savefig=True,format='png')
    fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')

    df_filters = {'all':None,
              'hi_intact':{'pyasState':'hi','vnc_status':'intact'},
              'lo_intact':{'pyasState':'lo','vnc_status':'intact'},
              'hi_cut':{'pyasState':'hi','vnc_status':'cut'},
              'lo_cut':{'pyasState':'lo','vnc_status':'cut'},
    }

    for filt_key in df_filters.keys():
        dff = df_filters[filt_key]
        fig,ax = T.plot_probe_distribution(binwidth=2,bin_min=-500,bin_max=10,
                                df_filter=dff,index=T.df.loc[208:].index,savefig=False,format='png',
                                export=True)



if RECOMPUTE:
    sinq,T = run_compute(sinq,id)
    run_plots(T)
    sinq.drop_tables()

sinq.set_note(
    dayflycell = id,
    text= 'VNC cut',
    tags=['control','vnc_cut'],
)

## 241101_F2_C1 -  ppk-Gal4>Chr in WT

In [ ]:
id = '241101_F2_C1'

# Special compute
def run_compute(sinq,id):
    if id in sinq.df.index:
        T = sinq.restore_table(dayflycell=id,overwrite=True)
    else:
        T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
        T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
        T.assign_column_value('fiberLED','625_red',trial_min =1,write_to_hdf5=True)
        T.assign_column_value('vnc_status','intact',trial_min =1,trial_max =208,write_to_hdf5=True)
        T.assign_column_value('vnc_status','cut',trial_min =209,write_to_hdf5=True)
        sinq.add_table(table=T)
    T.exclude_trials()
    T.extract_trial_properties()
    T.find_successful_trials()
    return sinq,T

def run_plots(T):
    fig,ax = T.plot_outcomes(savefig=True,format='png')
    fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')

    df_filters = {'all':None,
              'hi_intact':{'pyasState':'hi','vnc_status':'intact'},
              'lo_intact':{'pyasState':'lo','vnc_status':'intact'},
              'hi_cut':{'pyasState':'hi','vnc_status':'cut'},
              'lo_cut':{'pyasState':'lo','vnc_status':'cut'},
    }


    for filt_key in df_filters.keys():
        dff = df_filters[filt_key]
        fig,ax = T.plot_probe_distribution(binwidth=2,bin_min=-500,bin_max=10,
                                df_filter=dff,index=T.df.loc[208:].index,savefig=False,format='png',
                                export=True)


if RECOMPUTE:
    sinq,T = run_compute(sinq,id)
    run_plots(T)
    sinq.drop_tables()

sinq.set_note(
    dayflycell = id,
    text= 'VNC cut',
    tags=['control','vnc_cut'],
)


## 241105_F1_C1 -  ppk-Gal4>Chr in WT

In [ ]:
id = '241105_F1_C1'

# Special compute
def run_compute(sinq,id):
    if id in sinq.df.index:
        T = sinq.restore_table(dayflycell=id,overwrite=False)
    else:
        T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
        T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
        T.assign_column_value('fiberLED','625_red',trial_min =1,write_to_hdf5=True)
        T.assign_column_value('vnc_status','intact',trial_min =1,write_to_hdf5=True)
        sinq.add_table(table=T)
    T.exclude_trials()
    T.extract_trial_properties()
    T.find_successful_trials()
    return sinq,T

def run_plots(T):
    fig,ax = T.plot_outcomes(savefig=True,format='png')
    fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')

    df_filters = {'all':None,
              'hi_intact':{'pyasState':'hi','vnc_status':'intact'},
              'lo_intact':{'pyasState':'lo','vnc_status':'intact'},
            #   'hi_cut':{'pyasState':'hi','vnc_status':'cut'},
            #   'lo_cut':{'pyasState':'lo','vnc_status':'cut'},
    }

    for filt_key in df_filters.keys():
        dff = df_filters[filt_key]
        fig,ax = T.plot_probe_distribution(binwidth=2,bin_min=-500,bin_max=10,
                                df_filter=dff,savefig=False,format='png',
                                export=True)
        

if RECOMPUTE:
    sinq,T = run_compute(sinq,id)
    run_plots(T)
    sinq.drop_tables()

sinq.set_note(
    dayflycell = id,
    text= 'VNC intact',
    tags=['control','vnc_cut'],
)

# Hot-Cell-Gal4 below foil experiment

## 241105_F2_C1 -  Hot-Cell-Gal4>Chr in WT

In [ ]:
id = '241105_F2_C1'

# Special compute
def run_compute(sinq,id):
    if id in sinq.df.index:
        T = sinq.restore_table(dayflycell=id,overwrite=True)
    else:
        T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
        T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
        T.assign_column_value('fiberLED','625_red',trial_min =1,write_to_hdf5=True)
        T.assign_column_value('vnc_status','intact',trial_min =1,trial_max =246,write_to_hdf5=True)
        T.assign_column_value('vnc_status','cut',trial_min =247,write_to_hdf5=True)
        sinq.add_table(table=T)
    T.exclude_trials()
    T.extract_trial_properties()
    T.find_successful_trials()
    return sinq,T

def run_plots(T):
    fig,ax = T.plot_outcomes(savefig=True,format='png')
    fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')

    df_filters = {'all':None,
              'hi_intact':{'pyasState':'hi','vnc_status':'intact'},
              'lo_intact':{'pyasState':'lo','vnc_status':'intact'},
              'hi_cut':{'pyasState':'hi','vnc_status':'cut'},
              'lo_cut':{'pyasState':'lo','vnc_status':'cut'},
    }

    for filt_key in df_filters.keys():
        dff = df_filters[filt_key]
        fig,ax = T.plot_probe_distribution(binwidth=2,bin_min=-500,bin_max=10,
                                df_filter=dff,savefig=False,format='png',
                                export=True)


if RECOMPUTE:
    sinq,T = run_compute(sinq,id)
    run_plots(T)
    sinq.drop_tables()

sinq.set_note(
    dayflycell = id,
    text= 'VNC cut',
    tags=['control','vnc_cut'],
)

## 241105_F3_C1 -  Hot-Cell-Gal4>Chr in WT

In [ ]:
id = '241105_F3_C1'

# Special compute
def run_compute(sinq,id):
    if id in sinq.df.index:
        T = sinq.restore_table(dayflycell=id,overwrite=True)
    else:
        T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
        T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
        T.assign_column_value('fiberLED','625_red',trial_min =1,write_to_hdf5=True)
        T.assign_column_value('vnc_status','intact',trial_min =1,trial_max =208,write_to_hdf5=True)
        T.assign_column_value('vnc_status','cut',trial_min =209,write_to_hdf5=True)
        sinq.add_table(table=T)
    T.exclude_trials()
    T.extract_trial_properties()
    T.find_successful_trials()
    return sinq,T

def run_plots(T):
    fig,ax = T.plot_outcomes(savefig=True,format='png')
    fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')

    df_filters = {'all':None,
              'hi_intact':{'pyasState':'hi','vnc_status':'intact'},
              'lo_intact':{'pyasState':'lo','vnc_status':'intact'},
              'hi_cut':{'pyasState':'hi','vnc_status':'cut'},
              'lo_cut':{'pyasState':'lo','vnc_status':'cut'},
    }


    for filt_key in df_filters.keys():
        dff = df_filters[filt_key]
        fig,ax = T.plot_probe_distribution(binwidth=2,bin_min=-500,bin_max=10,
                                df_filter=dff,savefig=False,format='png',
                                export=True)



if not id in sinq.df.index or RECOMPUTE:
    sinq,T = run_compute(sinq,id)
    run_plots(T)
    sinq.drop_tables()

sinq.set_note(
    dayflycell = id,
    text= 'VNC cut',
    tags=['control','vnc_cut'],
    append=False
)

# Extension

In [ ]:
id = '260619_F1_C1'

T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
sinq.add_table(table=T)

# Extension plotting
fig,ax = T.plot_outcomes(savefig=True,format='png')
fig,ax,df = T.plot_probe_position_heatmap(cmin=-10,cmax=500,format='png')

df_filters = {'all':None,
              'hi':{'pyasState':'hi'},
              'lo':{'pyasState':'lo'},}

for filt_key in df_filters.keys():
    dff = df_filters[filt_key]
    fig,ax = T.plot_probe_distribution(binwidth=2,bin_min=-10,bin_max=500,
                            df_filter=dff,index=None,savefig=False,format='png',
                            export=True)
    
sinq.drop_tables()

sinq.set_note(
    dayflycell = id,
    text= 'Extension task',
    tags=['control','extension'],
    append=False
)

In [ ]:
id = '260619_F2_C1'

T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
sinq.add_table(table=T)


# run_plots(T)

# Extension plotting
fig,ax = T.plot_outcomes(savefig=True,format='png')
fig,ax,df = T.plot_probe_position_heatmap(cmin=-10,cmax=500,format='png')

df_filters = {'all':None,
              'hi':{'pyasState':'hi'},
              'lo':{'pyasState':'lo'},}

for filt_key in df_filters.keys():
    dff = df_filters[filt_key]
    fig,ax = T.plot_probe_distribution(binwidth=2,bin_min=-10,bin_max=500,
                            df_filter=dff,index=None,savefig=False,format='png',
                            export=True)
    

sinq.drop_tables()

sinq.set_note(
    dayflycell = id,
    text= 'Extension task',
    tags=['control','extension'],
    append=False
)

In [ ]:
id = '260623_F1_C1'

T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
sinq.add_table(table=T)


# run_plots(T)

# Extension plotting
fig,ax = T.plot_outcomes(savefig=True,format='png')
fig,ax,df = T.plot_probe_position_heatmap(cmin=-10,cmax=500,format='png')

df_filters = {'all':None,
              'hi':{'pyasState':'hi'},
              'lo':{'pyasState':'lo'},}

for filt_key in df_filters.keys():
    dff = df_filters[filt_key]
    fig,ax = T.plot_probe_distribution(binwidth=2,bin_min=-10,bin_max=500,
                            df_filter=dff,index=None,savefig=False,format='png',
                            export=True)
    

sinq.drop_tables()

sinq.set_note(
    dayflycell = id,
    text= 'Extension task',
    tags=['control','extension'],
    append=False
)

In [ ]:
id = '260623_F2_C1'

T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
sinq.add_table(table=T)


# run_plots(T)

# Extension plotting
fig,ax = T.plot_outcomes(savefig=True,format='png')
fig,ax,df = T.plot_probe_position_heatmap(cmin=-10,cmax=500,format='png')

df_filters = {'all':None,
              'hi':{'pyasState':'hi'},
              'lo':{'pyasState':'lo'},}

for filt_key in df_filters.keys():
    dff = df_filters[filt_key]
    fig,ax = T.plot_probe_distribution(binwidth=2,bin_min=-10,bin_max=500,
                            df_filter=dff,index=None,savefig=False,format='png',
                            export=True)
    

sinq.drop_tables()

sinq.set_note(
    dayflycell = id,
    text= 'Extension task',
    tags=['control','extension'],
    append=False
)

In [ ]:
T._excluded_df.head(30)

In [ ]:
id = '260623_F3_C1'

# T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
# T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)

# endofdaytrials = T.df.loc[492:]
# T.exclude_list_of_trials(endofdaytrials.index,reason='got past probe and could flex')
# T.exclude_trials()

# funky = T.df.loc[284:291]
# T.exclude_list_of_trials(funky.index,reason='bad probe illumination and flex')
# T.exclude_trials()
# T.extract_trial_properties()
# T.find_successful_trials()

# sinq.add_table(table=T)

# Extension plotting
# fig,ax = T.plot_outcomes(savefig=True,format='png')
# fig,ax,df = T.plot_probe_position_heatmap(cmin=-10,cmax=500,format='png')

# df_filters = {'all':None,
#               'hi':{'pyasState':'hi'},
#               'lo':{'pyasState':'lo'},}

# for filt_key in df_filters.keys():
#     dff = df_filters[filt_key]
#     fig,ax = T.plot_probe_distribution(binwidth=2,bin_min=-10,bin_max=500,
#                             df_filter=dff,index=None,savefig=False,format='png',
#                             export=True)
    

# sinq.drop_tables()

sinq.set_note(
    dayflycell = id,
    text= 'Extension task',
    tags=['control','extension'],
    append=False
)